# Catch the Liar v6
**New over v5:**
- Hallucination-targeted features (vagueness, informality, entity specificity, topic drift, coherence)
- Iterative pseudo-labeling on test set (self-training)
- GroupKFold respecting augmentation pairing (no swap leakage)
- CatBoost in stacking ensemble
- Rank-averaging meta-learner
- Bigger scratch GloVe (dim=80, vocab=8000)

In [1]:
import os, re, math, json, random, warnings, zlib
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter, defaultdict

from sklearn.model_selection import (RepeatedStratifiedKFold, StratifiedKFold,
                                      StratifiedGroupKFold, cross_val_score)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif, VarianceThreshold
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
from sklearn.decomposition import TruncatedSVD
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings('ignore')
random.seed(42); np.random.seed(42)

DATA_ROOT  = Path('fake-or-real-the-impostor-hunt/data')
TRAIN_DIR  = DATA_ROOT/'train'
TEST_DIR   = DATA_ROOT/'test'
LABELS_CSV = DATA_ROOT/'train.csv'
SAMPLE_SUB = DATA_ROOT/'sample_submission.csv'
ARXIV_PATH = Path('arxiv-metadata-oai-snapshot.json')
OUTPUT_DIR = Path('.')
print('Data root:', DATA_ROOT.exists(), '| arXiv:', ARXIV_PATH.exists())

Data root: True | arXiv: True


In [2]:
def load_pair(folder):
    return ((folder/'file_1.txt').read_text(encoding='utf-8',errors='replace').strip(),
            (folder/'file_2.txt').read_text(encoding='utf-8',errors='replace').strip())

def load_split(split_dir, labels_df=None):
    records=[]
    for folder in sorted(split_dir.iterdir()):
        if not folder.is_dir(): continue
        try: num_id=str(int(folder.name.split('_')[-1]))
        except: continue
        t1,t2=load_pair(folder)
        rec={'id':num_id,'text_1':t1,'text_2':t2}
        if labels_df is not None:
            row=labels_df[labels_df['id'].astype(str)==num_id]
            rec['label']=int(row['label'].values[0]) if len(row) else None
        records.append(rec)
    return pd.DataFrame(records)

labels_df=pd.read_csv(LABELS_CSV)
train_df=load_split(TRAIN_DIR,labels_df)
test_df=load_split(TEST_DIR)
print(f'Train:{len(train_df)} Test:{len(test_df)} Labels:{labels_df["label"].value_counts().to_dict()}')

Train:95 Test:1068 Labels:{2: 49, 1: 46}


In [3]:
def load_arxiv_astro(path, n=10000):
    texts=[]
    if not path.exists(): print('arXiv not found — train-only fallback'); return []
    with open(path) as f:
        for line in f:
            try:
                p=json.loads(line)
                if 'astro-ph' in p.get('categories','') and p.get('abstract','').strip():
                    texts.append(p['abstract'].replace('\n',' ').strip())
            except: pass
            if len(texts)>=n: break
    print(f'arXiv abstracts loaded: {len(texts)}')
    return texts

arxiv_texts=load_arxiv_astro(ARXIV_PATH, n=10000)

arXiv abstracts loaded: 10000


In [4]:
def tokenize(text): return re.findall(r"[a-z]+(?:'[a-z]+)?", text.lower())
def sentence_split(text):
    return [s for s in re.split(r'(?<=[.!?])\s+', text.strip()) if s.strip()]

In [5]:
class CharNgramLM:
    def __init__(self, n=4, k=0.05):
        self.n=n; self.k=k; self.counts=defaultdict(Counter); self.vocab=set()
    def _pad(self,t): return '<'*(self.n-1)+t.lower()+'>'
    def fit(self, texts):
        for t in texts:
            p=self._pad(t); self.vocab.update(p)
            for i in range(len(p)-self.n+1):
                self.counts[p[i:i+self.n-1]][p[i+self.n-1]]+=1
        return self
    def perplexity(self, text):
        p=self._pad(text); V=max(len(self.vocab),1); lp=0.0; N=0
        for i in range(len(p)-self.n+1):
            ctx=p[i:i+self.n-1]; ch=p[i+self.n-1]
            cc=self.counts.get(ctx,Counter()); ct=sum(cc.values())
            lp+=math.log2(max((cc[ch]+self.k)/(ct+self.k*V),1e-30)); N+=1
        return 2**(-lp/max(N,1))

class MultiNCharLM:
    def __init__(self, ns=(3,4,5), k=0.05):
        self.lms=[CharNgramLM(n=n,k=k) for n in ns]
    def fit(self, texts):
        for lm in self.lms: lm.fit(texts)
        return self
    def perplexity(self, text):
        ppls=[lm.perplexity(text) for lm in self.lms]
        return float(np.mean(ppls)), ppls

class KneserNeyBigramLM:
    def __init__(self, d=0.75):
        self.d=d; self.bigram=defaultdict(Counter)
        self.continuation=Counter(); self.total_continuation=0
    def fit(self, texts):
        left_ctx=defaultdict(set)
        for t in texts:
            toks=['<s>']+tokenize(t)+['</s>']
            for i in range(len(toks)-1):
                self.bigram[toks[i]][toks[i+1]]+=1
                left_ctx[toks[i+1]].add(toks[i])
        for w,ctxs in left_ctx.items(): self.continuation[w]=len(ctxs)
        self.total_continuation=max(sum(self.continuation.values()),1)
        return self
    def _pkn(self,w): return self.continuation.get(w,1)/self.total_continuation
    def _p(self,w,ctx):
        c_ctx=sum(self.bigram[ctx].values())
        if c_ctx==0: return self._pkn(w)
        lam=(self.d*len(self.bigram[ctx]))/c_ctx
        return max(self.bigram[ctx].get(w,0)-self.d,0)/c_ctx+lam*self._pkn(w)
    def perplexity(self, text):
        toks=['<s>']+tokenize(text)+['</s>']
        if len(toks)<2: return 100.0
        lp=sum(math.log2(max(self._p(toks[i],toks[i-1]),1e-30)) for i in range(1,len(toks)))
        return 2**(-lp/(len(toks)-1))

print('LM classes defined.')

LM classes defined.


In [6]:
class ScratchGloVe:
    """PPMI-SVD word embeddings trained from scratch."""
    def __init__(self, dim=80, window=5, min_count=3, vocab_size=8000):
        self.dim=dim; self.window=window
        self.min_count=min_count; self.vocab_size=vocab_size
        self.word2idx={}; self.embeddings=None

    def fit(self, texts):
        wf=Counter()
        for t in texts: wf.update(tokenize(t))
        vocab=[w for w,c in wf.most_common(self.vocab_size) if c>=self.min_count]
        self.word2idx={w:i for i,w in enumerate(vocab)}
        V=len(vocab)
        cooc=defaultdict(float)
        for t in texts:
            toks=[w for w in tokenize(t) if w in self.word2idx]
            for i,w in enumerate(toks):
                wi=self.word2idx[w]
                for j in range(max(0,i-self.window), min(len(toks),i+self.window+1)):
                    if i==j: continue
                    wj=self.word2idx[toks[j]]
                    cooc[(wi,wj)]+=1.0/abs(i-j)
        word_counts=np.zeros(V); ctx_counts=np.zeros(V)
        for (wi,wj),c in cooc.items():
            word_counts[wi]+=c; ctx_counts[wj]+=c
        total=max(sum(word_counts),1)
        rows,cols,vals=[],[],[]
        for (wi,wj),c in cooc.items():
            pmi=math.log(c*total/max(word_counts[wi]*ctx_counts[wj],1e-10))
            ppmi=max(pmi,0)
            if ppmi>0: rows.append(wi); cols.append(wj); vals.append(ppmi)
        from scipy.sparse import csr_matrix
        M=csr_matrix((vals,(rows,cols)),shape=(V,V))
        svd=TruncatedSVD(n_components=self.dim, random_state=42)
        self.embeddings=svd.fit_transform(M)
        norms=np.linalg.norm(self.embeddings,axis=1,keepdims=True)
        self.embeddings/=(norms+1e-10)
        print(f'GloVe vocab:{V} dim:{self.dim}')
        return self

    def text_vector(self, text):
        toks=[w for w in tokenize(text) if w in self.word2idx]
        if not toks: return np.zeros(self.dim)
        return np.array([self.embeddings[self.word2idx[w]] for w in toks]).mean(0)

    def wmd_proxy(self, t1, t2):
        v1=self.text_vector(t1); v2=self.text_vector(t2)
        n1,n2=np.linalg.norm(v1),np.linalg.norm(v2)
        return 1.0-float(np.dot(v1,v2)/(n1*n2)) if n1>0 and n2>0 else 1.0

print('ScratchGloVe defined.')

ScratchGloVe defined.


In [7]:
real_texts,fake_texts=[],[]
for _,row in train_df.iterrows():
    if row['label']==1: real_texts.append(row['text_1']); fake_texts.append(row['text_2'])
    elif row['label']==2: real_texts.append(row['text_2']); fake_texts.append(row['text_1'])

corpus=arxiv_texts+real_texts

print('Training GloVe...')
glove=ScratchGloVe(dim=80, window=5, min_count=3, vocab_size=8000).fit(corpus)

class ScratchTFIDF:
    def __init__(self, max_features=4000):
        self.max_features=max_features; self.vocab={}; self.idf=None
    def fit(self, corp):
        N=len(corp); df=Counter(); aw=Counter()
        for t in corp: toks=set(tokenize(t)); df.update(toks); aw.update(tokenize(t))
        sel=[w for w,_ in aw.most_common() if df[w]>1][:self.max_features]
        self.vocab={w:i for i,w in enumerate(sel)}
        self.idf=np.array([math.log((N+1)/(df[w]+1))+1 for w in sel])
        return self
    def transform(self, text):
        toks=tokenize(text); tf=Counter(toks)
        vec=np.zeros(len(self.vocab)); tot=max(len(toks),1)
        for w,i in self.vocab.items():
            if tf[w]>0: vec[i]=(tf[w]/tot)*self.idf[i]
        n=np.linalg.norm(vec); return vec/n if n>0 else vec

def cosine(a,b):
    na,nb=np.linalg.norm(a),np.linalg.norm(b)
    return float(np.dot(a,b)/(na*nb)) if na>0 and nb>0 else 0.0

tfidf_corpus=corpus+list(train_df['text_1'])+list(train_df['text_2'])
tfidf=ScratchTFIDF(max_features=4000).fit(tfidf_corpus)
print(f'TFIDF vocab:{len(tfidf.vocab)}')

real_vecs=np.array([tfidf.transform(t) for t in real_texts])
real_centroid=real_vecs.mean(0); real_centroid/=(np.linalg.norm(real_centroid)+1e-10)

n_lsa=min(60,len(tfidf.vocab)-1)
lsa=TruncatedSVD(n_components=n_lsa,random_state=42)
lsa.fit(np.array([tfidf.transform(t) for t in tfidf_corpus[:600]]))
print(f'LSA explained var:{lsa.explained_variance_ratio_.sum():.4f}')

real_glove=np.array([glove.text_vector(t) for t in real_texts])
real_glove_centroid=real_glove.mean(0)
rn=np.linalg.norm(real_glove_centroid); real_glove_centroid/=(rn+1e-10)

# Train LMs
lm_char_real=MultiNCharLM(ns=(3,4,5)).fit(corpus)
lm_char_fake=MultiNCharLM(ns=(3,4,5)).fit(fake_texts)
lm_kn_real=KneserNeyBigramLM().fit(corpus)
lm_kn_fake=KneserNeyBigramLM().fit(fake_texts)
print('All models trained.')

Training GloVe...
GloVe vocab:8000 dim:80
TFIDF vocab:4000
LSA explained var:0.2739
All models trained.


In [8]:
HEDGE={'approximately','roughly','about','around','nearly','almost','may','might','could',
       'possibly','perhaps','presumably','likely','seemingly','apparently','reportedly',
       'suggest','suggests','indicate','indicates','appear','appears','seem','seems',
       'generally','typically','usually','often','sometimes','occasionally'}
PREPS={'in','on','at','to','for','of','with','by','from','as','into','through','during',
       'before','after','above','below','between','among','under','over','within','without','against'}
DETS={'the','a','an','this','that','these','those','each','every'}
CONJS={'and','or','but','nor','yet','so','although','because','since','while','if',
       'unless','however','therefore'}
DISC={'however','therefore','consequently','furthermore','moreover','nevertheless',
      'nonetheless','meanwhile','subsequently','additionally'}

# NEW: Hallucination-targeted word lists
VAGUE={'various','many','some','several','numerous','vast','significant','important',
       'valuable','interesting','great','large','small','things','stuff','different',
       'specific','particular','certain','overall','general','notable','key','major',
       'essentially','basically','simply','just','really','very','quite','rather',
       'pretty','extremely','incredibly','remarkably','particularly','especially'}
INFORMAL={'hopefully','okay','ok','cool','awesome','amazing','incredible','fantastic',
          'definitely','absolutely','totally','honestly','actually','anyway','anyways',
          'gonna','wanna','gotta','kinda','sorta','btw','fyi','wow','lol','yeah'}
FUNCTION_WORDS=PREPS|DETS|CONJS|{'is','are','was','were','be','been','being','have',
    'has','had','do','does','did','will','would','shall','should','can','could',
    'may','might','must','not','no','it','its','they','them','their','we','our',
    'he','she','his','her','you','your','i','my','me','us','who','which','that',
    'this','these','those','what','where','when','how','why'}

def syllables(w): return max(1,len(re.findall(r'[aeiou]+',w)))

def text_features(text):
    """Extract surface + hallucination-targeted features from a single text."""
    words=tokenize(text); sents=sentence_split(text)

    _keys=['word_count','char_count','sent_count','avg_word_len','avg_sent_len',
           'sent_len_var','ttr','ttr_200','numeric_density','punct_density',
           'uppercase_ratio','hedge_density','prep_ratio','det_ratio','conj_ratio',
           'discourse_density','ne_count','flesch','long_word_ratio','char_entropy',
           'compression_ratio','self_repetition','bigram_entropy',
           'avg_syllables','hapax_ratio','sent_len_max','sent_len_min','trigram_entropy',
           # v6 NEW:
           'vagueness','informality_words','exclamation_count','question_count',
           'ellipsis_count','informal_punct_ratio',
           'acronym_density','exact_number_density','unit_number_density',
           'quoted_terms','parenthetical_defs','max_word_len',
           'unique_4gram_ratio','digit_char_ratio','para_count',
           'burstiness','zipf_coeff']
    if not words: return {k:0.0 for k in _keys}

    wc=len(words); cc=len(text); sc=len(sents); wf=Counter(words)
    swc=[len(tokenize(s)) for s in sents]
    asl=np.mean(swc) if swc else 0
    slv=np.var(swc) if swc else 0
    slmax=max(swc) if swc else 0
    slmin=min(swc) if swc else 0

    ttr=len(set(words))/wc
    ttr200=len(set(words[:200]))/min(200,wc)
    hapax=sum(1 for w,c in wf.items() if c==1)/wc

    nums=re.findall(r'\b\d+(?:\.\d+)?\b',text); nd=len(nums)/wc
    puncts=re.findall(r'[^\w\s]',text); pd_=len(puncts)/max(cc,1)
    raw=text.split()
    upper=[w for w in raw if w and (w[0].isupper() or w.isupper())]
    asw=sum(syllables(w) for w in words)/wc
    flesch=206.835-1.015*asl-84.6*asw if sents else 0

    cf=Counter(text.lower()); tc=sum(cf.values())
    ce=-sum((v/tc)*math.log2(v/tc) for v in cf.values() if v>0)

    enc=text.encode('utf-8')
    comp_r=len(zlib.compress(enc,9))/len(enc) if enc else 1.0

    # Self-repetition (sentence-level cosine)
    sent_vecs=[]
    for s in sents:
        t2=tokenize(s)
        if t2:
            v=Counter(t2); n=math.sqrt(sum(x*x for x in v.values()))
            sent_vecs.append({w:c/n for w,c in v.items()})
    self_r=0.0
    if len(sent_vecs)>=2:
        sims=[sum(sent_vecs[i].get(w,0)*sent_vecs[j].get(w,0)
                  for w in set(sent_vecs[i])&set(sent_vecs[j]))
              for i in range(len(sent_vecs)) for j in range(i+1,len(sent_vecs))]
        self_r=float(np.mean(sims))

    # N-gram entropies
    bigrams=list(zip(words[:-1],words[1:]))
    bg_ent=0.0
    if bigrams:
        bf=Counter(bigrams); bt=sum(bf.values())
        bg_ent=-sum((v/bt)*math.log2(v/bt) for v in bf.values() if v>0)
    trigrams=list(zip(words[:-2],words[1:-1],words[2:]))
    tg_ent=0.0
    if trigrams:
        tf_=Counter(trigrams); tt=sum(tf_.values())
        tg_ent=-sum((v/tt)*math.log2(v/tt) for v in tf_.values() if v>0)

    # ===== v6 NEW FEATURES =====

    # Vagueness score
    vagueness=sum(1 for w in words if w in VAGUE)/wc

    # Informality
    informality_w=sum(1 for w in words if w in INFORMAL)/wc
    excl_count=text.count('!')
    quest_count=text.count('?')
    ellipsis_count=text.count('...')
    informal_punct=(excl_count+ellipsis_count)/max(cc,1)

    # Entity specificity
    acronym_density=len(re.findall(r'\b[A-Z]{2,6}\b',text))/max(wc,1)
    exact_num_density=len(re.findall(r'\b\d+\.\d+\b',text))/max(wc,1)
    unit_num_density=len(re.findall(
        r'\b\d+(?:\.\d+)?\s*(?:km|pc|AU|Mpc|kpc|GHz|MHz|TB|PB|GB|MB|keV|MeV|GeV|Msun|Lsun|yr|arcsec|arcmin|deg)\b',
        text))/max(wc,1)
    quoted_terms=len(re.findall(r'"[^"]{2,40}"',text))/max(wc,1)
    paren_defs=len(re.findall(r'\([A-Z][^)]{5,60}\)',text))/max(wc,1)
    max_wl=max(len(w) for w in words)

    # Repetition: unique 4-gram ratio
    fourgrams=list(zip(words[:-3],words[1:-2],words[2:-1],words[3:]))
    u4=len(set(fourgrams))/max(len(fourgrams),1) if fourgrams else 1.0

    # Digit character ratio
    digit_chars=sum(1 for c in text if c.isdigit())
    digit_ratio=digit_chars/max(cc,1)

    # Paragraph count (split by double newline or single newline)
    paras=[p for p in re.split(r'\n\s*\n|\n', text) if p.strip()]
    para_count=len(paras)

    # Burstiness: std/mean of sentence lengths (LLM text is unnaturally uniform)
    burstiness=np.std(swc)/max(np.mean(swc),1e-6) if len(swc)>=2 else 0.0

    # Zipf coefficient: slope of log-rank vs log-freq (LLM deviates from natural Zipf)
    zipf=0.0
    if len(wf)>=5:
        ranked=sorted(wf.values(), reverse=True)
        n_fit=min(50,len(ranked))
        log_rank=np.log(np.arange(1,n_fit+1))
        log_freq=np.log(np.array(ranked[:n_fit], dtype=float))
        if np.std(log_rank)>0:
            zipf=-np.polyfit(log_rank,log_freq,1)[0]

    return {'word_count':wc,'char_count':cc,'sent_count':sc,
            'avg_word_len':np.mean([len(w) for w in words]),
            'avg_sent_len':asl,'sent_len_var':slv,
            'ttr':ttr,'ttr_200':ttr200,
            'numeric_density':nd,'punct_density':pd_,
            'uppercase_ratio':len(upper)/max(len(raw),1),
            'hedge_density':sum(1 for w in words if w in HEDGE)/wc,
            'prep_ratio':sum(1 for w in words if w in PREPS)/wc,
            'det_ratio':sum(1 for w in words if w in DETS)/wc,
            'conj_ratio':sum(1 for w in words if w in CONJS)/wc,
            'discourse_density':sum(1 for w in words if w in DISC)/wc,
            'ne_count':len(set(re.findall(r'(?<![.!?]\s)\b([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)\b',text))
                         |set(re.findall(r'\b[A-Z]{2,}\b',text)))/max(wc,1),
            'flesch':flesch,'long_word_ratio':sum(1 for w in words if len(w)>=7)/wc,
            'char_entropy':ce,'compression_ratio':comp_r,'self_repetition':self_r,
            'bigram_entropy':bg_ent,'avg_syllables':asw,'hapax_ratio':hapax,
            'sent_len_max':slmax,'sent_len_min':slmin,'trigram_entropy':tg_ent,
            # v6 new:
            'vagueness':vagueness,'informality_words':informality_w,
            'exclamation_count':excl_count,'question_count':quest_count,
            'ellipsis_count':ellipsis_count,'informal_punct_ratio':informal_punct,
            'acronym_density':acronym_density,'exact_number_density':exact_num_density,
            'unit_number_density':unit_num_density,
            'quoted_terms':quoted_terms,'parenthetical_defs':paren_defs,
            'max_word_len':max_wl,'unique_4gram_ratio':u4,
            'digit_char_ratio':digit_ratio,'para_count':para_count,
            'burstiness':burstiness,'zipf_coeff':zipf}

print(f'text_features() → {len(text_features("test text sample."))} features per text')

text_features() → 45 features per text


In [9]:
def js_div(a,b,k=500):
    wa=Counter(tokenize(a)); wb=Counter(tokenize(b))
    vocab=set(wa)|set(wb)
    if len(vocab)>k: vocab={w for w,_ in (wa+wb).most_common(k)}
    ta=sum(wa.values()) or 1; tb=sum(wb.values()) or 1
    pa=np.array([wa.get(w,0)/ta for w in vocab])
    pb=np.array([wb.get(w,0)/tb for w in vocab])
    m=0.5*(pa+pb)
    kl=lambda p,q: np.sum(p[((p>0)&(q>0))]*np.log2(p[((p>0)&(q>0))]/q[((p>0)&(q>0))]))
    return 0.5*kl(pa,m)+0.5*kl(pb,m)

def renyi_div(a,b,alpha=2.0,k=300):
    wa=Counter(tokenize(a)); wb=Counter(tokenize(b))
    vocab=set(wa)|set(wb)
    if len(vocab)>k: vocab={w for w,_ in (wa+wb).most_common(k)}
    eps=1e-10; ta=sum(wa.values()) or 1; tb=sum(wb.values()) or 1
    pa=np.array([max(wa.get(w,0)/ta,eps) for w in vocab])
    pb=np.array([max(wb.get(w,0)/tb,eps) for w in vocab])
    pa/=pa.sum(); pb/=pb.sum()
    return float((1/(alpha-1))*math.log2(max(np.sum(pa**alpha/pb**(alpha-1)),eps)))

def vocab_excl(a,b):
    sa=set(tokenize(a)); sb=set(tokenize(b))
    return len(sa-sb)/max(len(sa),1), len(sb-sa)/max(len(sb),1)

def ne_overlap(a,b):
    def nes(t): return (set(re.findall(r'(?<![.!?]\s)\b([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)\b',t))
                       |set(re.findall(r'\b[A-Z]{2,}\b',t))
                       |set(re.findall(r'\b\d+(?:\.\d+)?\s*(?:GHz|MHz|km|kg|m|s|yr|pc|AU|ly)?\b',t)))
    na,nb=nes(a),nes(b)
    if not na and not nb: return 1.0,1.0
    ov=len(na&nb)
    return ov/max(len(na),1), ov/max(len(nb),1)

def bigram_overlap(a,b):
    def bg(t): toks=tokenize(t); return set(zip(toks[:-1],toks[1:]))
    ba,bb=bg(a),bg(b)
    if not ba and not bb: return 1.0,1.0
    return len(ba&bb)/max(len(ba),1), len(ba&bb)/max(len(bb),1)

def numeric_consistency(a,b):
    na=set(re.findall(r'\b\d+(?:\.\d+)?\b',a))
    nb=set(re.findall(r'\b\d+(?:\.\d+)?\b',b))
    if not na and not nb: return 1.0,1.0
    ov=len(na&nb)
    return ov/max(len(na),1), ov/max(len(nb),1)

def sent_min_cosine(t1,t2):
    s1=sentence_split(t1); s2=sentence_split(t2)
    if not s1 or not s2: return 0.0,0.0
    def sv(s):
        t=tokenize(s); v=Counter(t); n=math.sqrt(sum(x*x for x in v.values()))
        return {w:c/n for w,c in v.items()} if n>0 else {}
    vecs2=[v for v in (sv(s) for s in s2) if v]
    if not vecs2: return 0.0,0.0
    scores=[]
    for s in s1:
        v1=sv(s)
        if not v1: continue
        best=max((sum(v1.get(w,0)*v2.get(w,0) for w in set(v1)&set(v2)) for v2 in vecs2), default=0.0)
        scores.append(best)
    if not scores: return 0.0,0.0
    return float(np.mean(scores)),float(min(scores))

def sent_ppl_stats(text, lm):
    sents=sentence_split(text)
    if len(sents)<2: p=lm.perplexity(text); return p,0.0,p,0.0
    ppls=[lm.perplexity(s) for s in sents if len(s)>10]
    if not ppls: p=lm.perplexity(text); return p,0.0,p,0.0
    mn=float(np.mean(ppls)); sd=float(np.std(ppls)); mx=float(np.max(ppls))
    cv=sd/max(mn,1e-6)
    return mn,sd,mx,cv

def pmi_vocab_overlap(a,b,top=200):
    wa=Counter(tokenize(a)); wb=Counter(tokenize(b))
    ta=sum(wa.values()) or 1; tb=sum(wb.values()) or 1
    shared=set(wa)&set(wb)
    if not shared: return 0.0
    pmi_scores=[]
    for w in list(shared)[:top]:
        pa=wa[w]/ta; pb=wb[w]/tb
        pmi_scores.append(math.log2(max(pa*pb,1e-20))-math.log2(max(pa,1e-20))-math.log2(max(pb,1e-20)))
    return float(np.mean(pmi_scores)) if pmi_scores else 0.0

# === v6 NEW cross-pair features ===

def content_word_overlap(t1, t2, min_len=5):
    """Jaccard of content words (proxy for nouns/adjectives). Hallucinated text drifts off-topic."""
    cw1={w for w in tokenize(t1) if len(w)>=min_len and w not in FUNCTION_WORDS}
    cw2={w for w in tokenize(t2) if len(w)>=min_len and w not in FUNCTION_WORDS}
    if not cw1 and not cw2: return 1.0, 0.0, 0.0
    jaccard=len(cw1&cw2)/max(len(cw1|cw2),1)
    excl1=len(cw1-cw2)/max(len(cw1),1)
    excl2=len(cw2-cw1)/max(len(cw2),1)
    return jaccard, excl1, excl2

def intra_coherence(text):
    """Cosine between consecutive sentences within a text. Hallucinations jump topics."""
    sents=sentence_split(text)
    if len(sents)<2: return 1.0, 1.0
    vecs=[tfidf.transform(s) for s in sents]
    consec=[cosine(vecs[i],vecs[i+1]) for i in range(len(vecs)-1)]
    consec=[c for c in consec if not np.isnan(c)]
    if not consec: return 1.0, 1.0
    return float(np.mean(consec)), float(np.min(consec))

def acronym_overlap(t1, t2):
    """Shared acronyms between pair. Real summaries of same article share acronyms."""
    a1=set(re.findall(r'\b[A-Z]{2,6}\b', t1))
    a2=set(re.findall(r'\b[A-Z]{2,6}\b', t2))
    if not a1 and not a2: return 1.0
    return len(a1&a2)/max(len(a1|a2),1)

print('Cross-pair features defined.')

Cross-pair features defined.


In [10]:
def pair_features(t1, t2, char_r=None, char_f=None, kn_r=None, kn_f=None):
    char_r=char_r or lm_char_real; char_f=char_f or lm_char_fake
    kn_r=kn_r or lm_kn_real; kn_f=kn_f or lm_kn_fake

    f1=text_features(t1); f2=text_features(t2)
    keys=sorted(f1.keys())
    v1=np.array([f1[k] for k in keys]); v2=np.array([f2[k] for k in keys])

    # Multi-n char LM perplexities
    cp1r_mean,cp1r_ns=char_r.perplexity(t1); cp2r_mean,cp2r_ns=char_r.perplexity(t2)
    cp1f_mean,_=char_f.perplexity(t1); cp2f_mean,_=char_f.perplexity(t2)
    clr1=cp1f_mean/max(cp1r_mean,1e-3); clr2=cp2f_mean/max(cp2r_mean,1e-3)

    # KN word LM perplexities
    kp1r=kn_r.perplexity(t1); kp2r=kn_r.perplexity(t2)
    kp1f=kn_f.perplexity(t1); kp2f=kn_f.perplexity(t2)
    klr1=kp1f/max(kp1r,1e-3); klr2=kp2f/max(kp2r,1e-3)

    # Sentence-level perplexity stats
    cs1m,cs1s,cs1x,cs1cv=sent_ppl_stats(t1,char_r.lms[1])
    cs2m,cs2s,cs2x,cs2cv=sent_ppl_stats(t2,char_r.lms[1])
    ks1m,ks1s,ks1x,ks1cv=sent_ppl_stats(t1,kn_r)
    ks2m,ks2s,ks2x,ks2cv=sent_ppl_stats(t2,kn_r)

    # TF-IDF + LSA
    vec1=tfidf.transform(t1); vec2=tfidf.transform(t2)
    cos1=cosine(vec1,real_centroid); cos2=cosine(vec2,real_centroid)
    cos12=cosine(vec1,vec2)
    lsa1=lsa.transform([vec1])[0]; lsa2=lsa.transform([vec2])[0]
    lsa_cos=cosine(lsa1,lsa2)

    # GloVe
    gv1=glove.text_vector(t1); gv2=glove.text_vector(t2)
    gcos1=cosine(gv1,real_glove_centroid); gcos2=cosine(gv2,real_glove_centroid)
    gcos12=cosine(gv1,gv2); wmd=glove.wmd_proxy(t1,t2)

    # v5 cross-pair
    js=js_div(t1,t2); ren=renyi_div(t1,t2)
    ex1,ex2=vocab_excl(t1,t2); ne1,ne2=ne_overlap(t1,t2)
    bg1,bg2=bigram_overlap(t1,t2); num1,num2=numeric_consistency(t1,t2)
    smean1,smin1=sent_min_cosine(t1,t2); smean2,smin2=sent_min_cosine(t2,t1)
    pmi12=pmi_vocab_overlap(t1,t2)

    # v6 NEW cross-pair
    cw_jaccard, cw_excl1, cw_excl2 = content_word_overlap(t1, t2)
    ic1_mean, ic1_min = intra_coherence(t1)
    ic2_mean, ic2_min = intra_coherence(t2)
    acr_ov = acronym_overlap(t1, t2)

    # Build per-text extended vectors
    ext1=np.array([cp1r_mean,*cp1r_ns,cp1f_mean,clr1,
                   kp1r,kp1f,klr1,
                   cs1m,cs1s,cs1x,cs1cv,ks1m,ks1s,ks1x,ks1cv,
                   cos1,ex1,ne1,bg1,num1,smean1,smin1,
                   gcos1,ic1_mean,ic1_min])
    ext2=np.array([cp2r_mean,*cp2r_ns,cp2f_mean,clr2,
                   kp2r,kp2f,klr2,
                   cs2m,cs2s,cs2x,cs2cv,ks2m,ks2s,ks2x,ks2cv,
                   cos2,ex2,ne2,bg2,num2,smean2,smin2,
                   gcos2,ic2_mean,ic2_min])

    full1=np.concatenate([v1,ext1,lsa1,gv1])
    full2=np.concatenate([v2,ext2,lsa2,gv2])
    delta=full1-full2
    absdelta=np.abs(delta)
    ratio=full1/(np.abs(full2)+1e-6)

    cross=np.array([
        js,ren,cos12,lsa_cos,gcos12,wmd,pmi12,
        cw_jaccard, cw_excl1, cw_excl2, acr_ov,  # v6 new
        cp1r_mean-cp2r_mean, cp1f_mean-cp2f_mean, clr1-clr2,
        kp1r-kp2r, kp1f-kp2f, klr1-klr2,
        cs1s-cs2s, cs1cv-cs2cv, ks1s-ks2s, ks1cv-ks2cv,
        cos1-cos2, gcos1-gcos2,
        bg1-bg2, ex1-ex2, ne1-ne2,
        num1-num2, smean1-smean2, smin1-smin2,
        ic1_mean-ic2_mean, ic1_min-ic2_min,  # v6 new
        f1['word_count']-f2['word_count'],
        f1['ttr']-f2['ttr'],
        f1['numeric_density']-f2['numeric_density'],
        f1['hapax_ratio']-f2['hapax_ratio'],
        f1['bigram_entropy']-f2['bigram_entropy'],
        f1['trigram_entropy']-f2['trigram_entropy'],
        f1['compression_ratio']-f2['compression_ratio'],
        # v6 new deltas:
        f1['vagueness']-f2['vagueness'],
        f1['informality_words']-f2['informality_words'],
        f1['exclamation_count']-f2['exclamation_count'],
        f1['acronym_density']-f2['acronym_density'],
        f1['exact_number_density']-f2['exact_number_density'],
        f1['unit_number_density']-f2['unit_number_density'],
        f1['unique_4gram_ratio']-f2['unique_4gram_ratio'],
        f1['burstiness']-f2['burstiness'],
        f1['zipf_coeff']-f2['zipf_coeff'],
        f1['informal_punct_ratio']-f2['informal_punct_ratio'],
    ])

    return np.concatenate([full1,full2,delta,absdelta,ratio,cross])

_demo=pair_features(train_df.iloc[0]['text_1'], train_df.iloc[0]['text_2'])
print(f'pair_features() → {len(_demo)} dims')

pair_features() → 1108 dims


In [11]:
print('Extracting test features...')
X_test=np.array([pair_features(r['text_1'],r['text_2']) for _,r in test_df.iterrows()])
X_test=np.nan_to_num(X_test,nan=0.0,posinf=0.0,neginf=0.0)
# Also compute swap features for test (needed for pseudo-label augmentation)
print('Extracting test swap features...')
X_test_swap=np.array([pair_features(r['text_2'],r['text_1']) for _,r in test_df.iterrows()])
X_test_swap=np.nan_to_num(X_test_swap,nan=0.0,posinf=0.0,neginf=0.0)
print(f'X_test:{X_test.shape} X_test_swap:{X_test_swap.shape}')

Extracting test features...
Extracting test swap features...
X_test:(1068, 1108) X_test_swap:(1068, 1108)


In [12]:
print('Extracting train features with LOO LMs...')
train_rows=[]; y_list=[]

for _,row in train_df.iterrows():
    label=row['label']; t1,t2=row['text_1'],row['text_2']
    ri=real_texts.index(t2 if label==2 else t1)
    fi=fake_texts.index(t1 if label==2 else t2)

    loo_r=[t for j,t in enumerate(real_texts) if j!=ri]
    loo_f=[t for j,t in enumerate(fake_texts) if j!=fi]

    cr=MultiNCharLM(ns=(3,4,5)).fit(arxiv_texts+loo_r)
    cf=MultiNCharLM(ns=(3,4,5)).fit(loo_f if loo_f else fake_texts)
    kr=KneserNeyBigramLM().fit(arxiv_texts+loo_r)
    kf=KneserNeyBigramLM().fit(loo_f if loo_f else fake_texts)

    fv=pair_features(t1,t2,cr,cf,kr,kf)
    train_rows.append(fv); y_list.append(int(label==2))
    # Augment: swap t1<->t2, flip label
    fv_swap=pair_features(t2,t1,cr,cf,kr,kf)
    train_rows.append(fv_swap); y_list.append(int(label==1))

X_train_orig=np.nan_to_num(np.array(train_rows),nan=0.0,posinf=0.0,neginf=0.0)
y_train_orig=np.array(y_list)
groups_orig=np.repeat(np.arange(len(train_df)),2)  # [0,0,1,1,...,94,94]
print(f'X_train:{X_train_orig.shape} balance:{y_train_orig.mean():.3f}')

Extracting train features with LOO LMs...
X_train:(190, 1108) balance:0.500


In [13]:
vt=VarianceThreshold(threshold=1e-6)
X_vt=vt.fit_transform(X_train_orig)
X_test_vt=vt.transform(X_test)
X_test_swap_vt=vt.transform(X_test_swap)
print(f'After variance filter: {X_vt.shape[1]} features')

K=min(250, X_vt.shape[1])
mi_sel=SelectKBest(mutual_info_classif, k=K)
X_sel=mi_sel.fit_transform(X_vt, y_train_orig)
X_test_sel=mi_sel.transform(X_test_vt)
X_test_swap_sel=mi_sel.transform(X_test_swap_vt)
print(f'After MI selection (k={K}): {X_sel.shape[1]} features')

# Scaler for LR/SVM
scaler=StandardScaler()
Xs=scaler.fit_transform(X_sel)
Xs_test=scaler.transform(X_test_sel)

After variance filter: 1107 features
After MI selection (k=250): 250 features


In [14]:
RSKF=RepeatedStratifiedKFold(n_splits=5,n_repeats=10,random_state=42)
SGKF5=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=42)

print('=== Baselines (RepeatedStratifiedKFold) ===')
for name,clf in [
    ('LR', LogisticRegression(C=1.0,solver='lbfgs',max_iter=2000,random_state=42)),
    ('SVM',SVC(C=1.0,kernel='rbf',probability=True,random_state=42)),
    ('RF', RandomForestClassifier(n_estimators=300,max_depth=5,random_state=42)),
    ('LGBM',lgb.LGBMClassifier(n_estimators=300,max_depth=3,learning_rate=0.05,
                                 num_leaves=15,random_state=42,verbose=-1)),
    ('CB', CatBoostClassifier(iterations=300,depth=4,learning_rate=0.05,
                               random_seed=42,verbose=0)),
]:
    inp=Xs if name in ('LR','SVM') else X_sel
    sc=cross_val_score(clf,inp,y_train_orig,cv=RSKF,scoring='accuracy')
    print(f'{name:5s} | CV:{sc.mean():.4f} ± {sc.std():.4f}')

print('\n=== Baselines (StratifiedGroupKFold — honest) ===')
for name,clf in [
    ('RF', RandomForestClassifier(n_estimators=300,max_depth=5,random_state=42)),
    ('LGBM',lgb.LGBMClassifier(n_estimators=300,max_depth=3,learning_rate=0.05,
                                 num_leaves=15,random_state=42,verbose=-1)),
]:
    sc=cross_val_score(clf,X_sel,y_train_orig,cv=SGKF5,scoring='accuracy',groups=groups_orig)
    print(f'{name:5s} | GroupCV:{sc.mean():.4f} ± {sc.std():.4f}')

=== Baselines (RepeatedStratifiedKFold) ===
LR    | CV:0.9000 ± 0.0450
SVM   | CV:0.8779 ± 0.0414
RF    | CV:0.9263 ± 0.0341
LGBM  | CV:0.9242 ± 0.0340
CB    | CV:0.9300 ± 0.0331

=== Baselines (StratifiedGroupKFold — honest) ===
RF    | GroupCV:0.9421 ± 0.0482
LGBM  | GroupCV:0.9263 ± 0.0453


In [15]:
def obj_xgb(trial):
    p={'max_depth':trial.suggest_int('max_depth',2,4),
       'min_child_weight':trial.suggest_int('min_child_weight',3,20),
       'gamma':trial.suggest_float('gamma',0.5,5.0),
       'n_estimators':trial.suggest_int('n_estimators',100,600),
       'learning_rate':trial.suggest_float('learning_rate',0.005,0.15,log=True),
       'subsample':trial.suggest_float('subsample',0.5,0.9),
       'colsample_bytree':trial.suggest_float('colsample_bytree',0.3,0.8),
       'colsample_bylevel':trial.suggest_float('colsample_bylevel',0.3,0.8),
       'colsample_bynode':trial.suggest_float('colsample_bynode',0.3,0.8),
       'reg_alpha':trial.suggest_float('reg_alpha',0.1,20.0,log=True),
       'reg_lambda':trial.suggest_float('reg_lambda',0.1,20.0,log=True),
       'eval_metric':'logloss','random_state':42,'n_jobs':-1}
    return cross_val_score(xgb.XGBClassifier(**p),X_sel,y_train_orig,
                           cv=SGKF5,scoring='accuracy',groups=groups_orig).mean()

def obj_lgb(trial):
    p={'max_depth':trial.suggest_int('max_depth',2,4),
       'num_leaves':trial.suggest_int('num_leaves',7,31),
       'min_child_samples':trial.suggest_int('min_child_samples',5,30),
       'n_estimators':trial.suggest_int('n_estimators',100,600),
       'learning_rate':trial.suggest_float('learning_rate',0.005,0.15,log=True),
       'subsample':trial.suggest_float('subsample',0.5,0.9),
       'colsample_bytree':trial.suggest_float('colsample_bytree',0.3,0.8),
       'reg_alpha':trial.suggest_float('reg_alpha',0.1,20.0,log=True),
       'reg_lambda':trial.suggest_float('reg_lambda',0.1,20.0,log=True),
       'random_state':42,'verbose':-1,'n_jobs':-1}
    return cross_val_score(lgb.LGBMClassifier(**p),X_sel,y_train_orig,
                           cv=SGKF5,scoring='accuracy',groups=groups_orig).mean()

def obj_cb(trial):
    p={'depth':trial.suggest_int('depth',2,5),
       'iterations':trial.suggest_int('iterations',100,600),
       'learning_rate':trial.suggest_float('learning_rate',0.005,0.15,log=True),
       'l2_leaf_reg':trial.suggest_float('l2_leaf_reg',0.1,20.0,log=True),
       'subsample':trial.suggest_float('subsample',0.5,0.9),
       'colsample_bylevel':trial.suggest_float('colsample_bylevel',0.3,0.8),
       'random_seed':42,'verbose':0}
    return cross_val_score(CatBoostClassifier(**p),X_sel,y_train_orig,
                           cv=SGKF5,scoring='accuracy',groups=groups_orig).mean()

print('Optuna XGB (150 trials)...')
sx=optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler(seed=42))
sx.optimize(obj_xgb, n_trials=150, show_progress_bar=True)
best_xgb={**sx.best_params,'eval_metric':'logloss','random_state':42,'n_jobs':-1}
print(f'XGB best CV:{sx.best_value:.4f}')

print('Optuna LGBM (100 trials)...')
sl=optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler(seed=0))
sl.optimize(obj_lgb, n_trials=100, show_progress_bar=True)
best_lgb={**sl.best_params,'random_state':42,'verbose':-1,'n_jobs':-1}
print(f'LGBM best CV:{sl.best_value:.4f}')

print('Optuna CatBoost (80 trials)...')
sc=optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler(seed=1))
sc.optimize(obj_cb, n_trials=80, show_progress_bar=True)
best_cb={**sc.best_params,'random_seed':42,'verbose':0}
print(f'CB best CV:{sc.best_value:.4f}')

Optuna XGB (150 trials)...


  0%|          | 0/150 [00:00<?, ?it/s]

XGB best CV:0.9368
Optuna LGBM (100 trials)...


  0%|          | 0/100 [00:00<?, ?it/s]

LGBM best CV:0.9263
Optuna CatBoost (80 trials)...


  0%|          | 0/80 [00:00<?, ?it/s]

CB best CV:0.9368


In [16]:
def run_stacking(X_tr, y_tr, groups, X_te, base_cfg, n_splits=5):
    """
    Train stacking ensemble, return OOF probs, test probs from each base model,
    and the meta-learner.
    """
    sgkf=StratifiedGroupKFold(n_splits=n_splits,shuffle=True,random_state=42)

    oof_probs=np.zeros((len(y_tr), len(base_cfg)))
    test_probs_all=np.zeros((len(X_te), len(base_cfg)))

    for ci,(name,make_clf) in enumerate(base_cfg):
        fold_test_probs=[]
        for tr_idx,val_idx in sgkf.split(X_tr, y_tr, groups):
            clf=make_clf()
            clf.fit(X_tr[tr_idx], y_tr[tr_idx])
            oof_probs[val_idx,ci]=clf.predict_proba(X_tr[val_idx])[:,1]
            fold_test_probs.append(clf.predict_proba(X_te)[:,1])
        # Average test predictions across folds
        test_probs_all[:,ci]=np.mean(fold_test_probs, axis=0)
        oof_acc=accuracy_score(y_tr, (oof_probs[:,ci]>=0.5).astype(int))
        print(f'  OOF {name}: acc={oof_acc:.4f}')

    # Meta-learner: LR on OOF probs
    meta=LogisticRegression(C=1.0,solver='lbfgs',max_iter=1000,random_state=42)
    meta.fit(oof_probs, y_tr)
    meta_oof=meta.predict_proba(oof_probs)[:,1]
    meta_test=meta.predict_proba(test_probs_all)[:,1]
    print(f'  Stacking OOF acc: {accuracy_score(y_tr, (meta_oof>=0.5).astype(int)):.4f}')

    # Rank averaging (more robust alternative)
    from scipy.stats import rankdata
    rank_oof=np.zeros_like(oof_probs)
    rank_test=np.zeros_like(test_probs_all)
    for ci in range(oof_probs.shape[1]):
        rank_oof[:,ci]=rankdata(oof_probs[:,ci])/len(oof_probs)
        rank_test[:,ci]=rankdata(test_probs_all[:,ci])/len(test_probs_all)
    rank_avg_oof=rank_oof.mean(axis=1)
    rank_avg_test=rank_test.mean(axis=1)

    return {
        'oof_probs': oof_probs,
        'meta_oof': meta_oof,
        'meta_test': meta_test,
        'rank_avg_oof': rank_avg_oof,
        'rank_avg_test': rank_avg_test,
        'meta': meta,
        'test_probs_all': test_probs_all,
    }

# Define base classifiers
base_cfg=[
    ('xgb', lambda: xgb.XGBClassifier(**best_xgb)),
    ('lgb', lambda: lgb.LGBMClassifier(**best_lgb)),
    ('cb',  lambda: CatBoostClassifier(**best_cb)),
    ('rf',  lambda: RandomForestClassifier(n_estimators=500,max_depth=5,
                                            min_samples_leaf=3,random_state=42)),
]

print('\n=== Initial Stacking (on original 190 samples) ===')
stack_result=run_stacking(X_sel, y_train_orig, groups_orig, X_test_sel, base_cfg)

# Optimal threshold on OOF
best_t,best_acc=0.5,0.0
for t in np.arange(0.1,0.9,0.01):
    acc=accuracy_score(y_train_orig, (stack_result['meta_oof']>=t).astype(int))
    if acc>best_acc: best_acc=acc; best_t=t
print(f'Optimal meta threshold: {best_t:.2f} → OOF acc: {best_acc:.4f}')

# Also check rank averaging
best_t_rank,best_acc_rank=0.5,0.0
for t in np.arange(0.1,0.9,0.01):
    acc=accuracy_score(y_train_orig, (stack_result['rank_avg_oof']>=t).astype(int))
    if acc>best_acc_rank: best_acc_rank=acc; best_t_rank=t
print(f'Optimal rank threshold: {best_t_rank:.2f} → OOF acc: {best_acc_rank:.4f}')


=== Initial Stacking (on original 190 samples) ===
  OOF xgb: acc=0.9368
  OOF lgb: acc=0.9263
  OOF cb: acc=0.9368
  OOF rf: acc=0.9211
  Stacking OOF acc: 0.9316
Optimal meta threshold: 0.24 → OOF acc: 0.9368
Optimal rank threshold: 0.45 → OOF acc: 0.9316


In [17]:
def pseudo_label_round(X_tr, y_tr, groups, X_te, X_te_swap, stack_result,
                       conf_threshold=0.85, base_cfg=base_cfg, round_num=1):
    """
    Take high-confidence test predictions, add them to training, retrain.
    Returns expanded X_tr, y_tr, groups, new stack_result.
    """
    # Use meta-learner probabilities for confidence
    test_probs = stack_result['meta_test']

    # High confidence: prob > conf_threshold (predict class 1) or prob < 1-conf_threshold (predict class 0)
    confident_pos = np.where(test_probs >= conf_threshold)[0]
    confident_neg = np.where(test_probs <= (1.0 - conf_threshold))[0]
    pseudo_idx = np.concatenate([confident_pos, confident_neg])
    pseudo_labels = np.concatenate([np.ones(len(confident_pos)), np.zeros(len(confident_neg))]).astype(int)

    print(f'\n  Round {round_num}: conf_threshold={conf_threshold:.2f}')
    print(f'  Pseudo-labeled: {len(pseudo_idx)} ({len(confident_pos)} pos, {len(confident_neg)} neg)')

    if len(pseudo_idx) < 10:
        print('  Too few pseudo-labels, skipping round.')
        return X_tr, y_tr, groups, stack_result

    # Add pseudo-labeled samples (both original and swapped for augmentation)
    pseudo_X = X_te[pseudo_idx]
    pseudo_X_swap = X_te_swap[pseudo_idx]
    pseudo_y = pseudo_labels
    pseudo_y_swap = 1 - pseudo_labels  # Flipped for swap

    # Create group IDs for pseudo-labeled samples
    max_group = groups.max() + 1
    pseudo_groups = np.repeat(np.arange(max_group, max_group + len(pseudo_idx)), 2)

    # Stack: original + pseudo (with swap augmentation)
    X_expanded = np.vstack([X_tr] + [np.vstack([pseudo_X[i:i+1], pseudo_X_swap[i:i+1]])
                                      for i in range(len(pseudo_idx))])
    y_expanded = np.concatenate([y_tr] + [np.array([pseudo_y[i], pseudo_y_swap[i]])
                                           for i in range(len(pseudo_idx))])
    groups_expanded = np.concatenate([groups, pseudo_groups])

    print(f'  Expanded training: {len(y_expanded)} samples (was {len(y_tr)})')
    print(f'  Balance: {y_expanded.mean():.3f}')

    # Retrain stacking on expanded set
    new_result = run_stacking(X_expanded, y_expanded, groups_expanded, X_te, base_cfg)
    return X_expanded, y_expanded, groups_expanded, new_result

# Run pseudo-labeling rounds
X_curr, y_curr, groups_curr = X_sel.copy(), y_train_orig.copy(), groups_orig.copy()
result_curr = stack_result

print('\n========== PSEUDO-LABELING ==========')
for rnd, thresh in enumerate([0.88, 0.82, 0.78], 1):
    X_curr, y_curr, groups_curr, result_curr = pseudo_label_round(
        X_curr, y_curr, groups_curr, X_test_sel, X_test_swap_sel,
        result_curr, conf_threshold=thresh, round_num=rnd)

# Final threshold optimization
best_t_final,best_acc_final=0.5,0.0
for t in np.arange(0.1,0.9,0.01):
    acc=accuracy_score(y_curr, (result_curr['meta_oof']>=t).astype(int))
    if acc>best_acc_final: best_acc_final=acc; best_t_final=t
print(f'\nFinal meta threshold: {best_t_final:.2f} → OOF acc: {best_acc_final:.4f}')
best_t_final = 0.45  # manual override

best_t_rank_final,best_acc_rank_final=0.5,0.0
for t in np.arange(0.1,0.9,0.01):
    acc=accuracy_score(y_curr, (result_curr['rank_avg_oof']>=t).astype(int))
    if acc>best_acc_rank_final: best_acc_rank_final=acc; best_t_rank_final=t
print(f'Final rank threshold: {best_t_rank_final:.2f} → OOF acc: {best_acc_rank_final:.4f}')
best_t_rank_final = 0.45  # manual override


========== PSEUDO-LABELING ==========

  Round 1: conf_threshold=0.88
  Pseudo-labeled: 837 (397 pos, 440 neg)
  Expanded training: 1864 samples (was 190)
  Balance: 0.500
  OOF xgb: acc=0.9925
  OOF lgb: acc=0.9930
  OOF cb: acc=0.9936
  OOF rf: acc=0.9936
  Stacking OOF acc: 0.9925

  Round 2: conf_threshold=0.82
  Pseudo-labeled: 1012 (485 pos, 527 neg)
  Expanded training: 3888 samples (was 1864)
  Balance: 0.500
  OOF xgb: acc=0.9933
  OOF lgb: acc=0.9956
  OOF cb: acc=0.9959
  OOF rf: acc=0.9964
  Stacking OOF acc: 0.9964

  Round 3: conf_threshold=0.78
  Pseudo-labeled: 1043 (495 pos, 548 neg)
  Expanded training: 5974 samples (was 3888)
  Balance: 0.500
  OOF xgb: acc=0.9925
  OOF lgb: acc=0.9980
  OOF cb: acc=0.9977
  OOF rf: acc=0.9970
  Stacking OOF acc: 0.9978

Final meta threshold: 0.56 → OOF acc: 0.9980
Final rank threshold: 0.50 → OOF acc: 0.9973


In [34]:
# Choose best method (meta LR vs rank averaging) based on OOF accuracy
if best_acc_final >= best_acc_rank_final:
    print(f'Using META-LEARNER (threshold={best_t_final:.2f})')
    test_final_probs = result_curr['meta_test']
    use_threshold = best_t_final
else:
    print(f'Using RANK AVERAGING (threshold={best_t_rank_final:.2f})')
    test_final_probs = result_curr['rank_avg_test']
    use_threshold = best_t_rank_final

# Also train final models on ALL expanded data and predict test
print('\nTraining final models on full expanded training set...')
final_test_probs = np.zeros((len(test_df), len(base_cfg)))
for ci,(name,make_clf) in enumerate(base_cfg):
    clf=make_clf()
    clf.fit(X_curr, y_curr)
    final_test_probs[:,ci] = clf.predict_proba(X_test_sel)[:,1]
    print(f'  {name} trained on {len(y_curr)} samples')

# Meta prediction on final
final_meta_probs = result_curr['meta'].predict_proba(final_test_probs)[:,1]

# Rank averaging on final
from scipy.stats import rankdata
rank_final = np.zeros_like(final_test_probs)
for ci in range(final_test_probs.shape[1]):
    rank_final[:,ci] = rankdata(final_test_probs[:,ci]) / len(final_test_probs)
final_rank_probs = rank_final.mean(axis=1)

# Use whichever performed better on OOF
if best_acc_final >= best_acc_rank_final:
    test_preds = (final_meta_probs >= use_threshold).astype(int)
    print(f'Final preds via meta LR, threshold={use_threshold:.2f}')
else:
    test_preds = (final_rank_probs >= use_threshold).astype(int)
    print(f'Final preds via rank avg, threshold={use_threshold:.2f}')

test_labels = test_preds + 1  # 0→1 (file1 real), 1→2 (file2 real)

print(f'Test label dist: {Counter(test_labels)}')
print(f'Test prob: mean={final_meta_probs.mean():.3f} std={final_meta_probs.std():.3f}')

# Submission
sub=pd.DataFrame({'id':test_df['id'].values, 'label':test_labels})
if SAMPLE_SUB.exists():
    sub=pd.read_csv(SAMPLE_SUB)[['id']].merge(sub, on='id', how='left')
sub.to_csv(OUTPUT_DIR/'submission_v6.csv', index=False)
print(f'\nSaved submission_v6.csv')
print(sub.head(10))
print(f'Label dist: {sub["label"].value_counts().to_dict()}')

Using META-LEARNER (threshold=0.45)

Training final models on full expanded training set...
  xgb trained on 5974 samples
  lgb trained on 5974 samples
  cb trained on 5974 samples
  rf trained on 5974 samples
Final preds via meta LR, threshold=0.45
Test label dist: Counter({np.int64(1): 562, np.int64(2): 506})
Test prob: mean=0.474 std=0.493

Saved submission_v6.csv
  id  label
0  0      2
1  1      2
2  2      1
3  3      1
4  4      2
5  5      1
6  6      2
7  7      1
8  8      2
9  9      1
Label dist: {1: 562, 2: 506}


## Diagnostic: Feature Importances
Run this to see which new features matter most

In [35]:
# Quick check: what features does the best XGB think are important?
clf_diag = xgb.XGBClassifier(**best_xgb)
clf_diag.fit(X_sel, y_train_orig)
imp = clf_diag.feature_importances_
top_k = 30
top_idx = np.argsort(imp)[-top_k:][::-1]
print(f'\nTop {top_k} feature indices and importances:')
for i,idx in enumerate(top_idx):
    print(f'  #{i+1}: feature[{idx}] = {imp[idx]:.4f}')


Top 30 feature indices and importances:
  #1: feature[220] = 0.0409
  #2: feature[194] = 0.0355
  #3: feature[227] = 0.0354
  #4: feature[7] = 0.0299
  #5: feature[222] = 0.0292
  #6: feature[111] = 0.0273
  #7: feature[118] = 0.0247
  #8: feature[112] = 0.0234
  #9: feature[95] = 0.0228
  #10: feature[187] = 0.0225
  #11: feature[186] = 0.0222
  #12: feature[183] = 0.0217
  #13: feature[114] = 0.0212
  #14: feature[121] = 0.0212
  #15: feature[224] = 0.0212
  #16: feature[249] = 0.0209
  #17: feature[192] = 0.0206
  #18: feature[189] = 0.0199
  #19: feature[181] = 0.0199
  #20: feature[235] = 0.0198
  #21: feature[169] = 0.0198
  #22: feature[156] = 0.0194
  #23: feature[116] = 0.0193
  #24: feature[68] = 0.0190
  #25: feature[92] = 0.0181
  #26: feature[21] = 0.0180
  #27: feature[29] = 0.0177
  #28: feature[198] = 0.0172
  #29: feature[113] = 0.0166
  #30: feature[125] = 0.0163
